# Supervised learning

**Track:** Foundations. **Level:** Foundations. **CPU:** yes.

## Goals

- Fit a classifier and a regressor and report the metric that matches the decision.
- See why accuracy is the wrong summary on an imbalanced problem.
- Estimate generalization with a held-out split and with cross-validation.
- Produce a model that memorizes the training set.

## Sources and license

Original notebook following the scikit-learn supervised tutorials (BSD-3): classification metrics, cross-validation, and the overfitting discussion in the user guide. The API calls follow that documentation. The datasets are synthetic, from `sklearn.datasets`.

## Concepts

A supervised model sees pairs `(x, y)` and is scored on pairs it did not fit. Accuracy is the fraction of labels that match. Precision is the fraction of predicted positives that are positive. Recall is the fraction of real positives that were found. F1 is their harmonic mean. ROC-AUC ranks scores and does not need a threshold. For regression, mean squared error is in units of `y` squared and R² compares the model with predicting the mean. Cross-validation averages a metric over folds so one lucky split is not the whole story. Overfitting is a training score that is far better than the test score.


In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 'numpy', 'pandas', 'scikit-learn', 'matplotlib'])
print("packages ready")


In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "labs" / "_shared" / "llm.py").exists():
            return cand
    if "google.colab" in sys.modules:
        root = Path("/content/ShaiLearnsAI")
        marker = root / "labs" / "_shared" / "llm.py"
        if not marker.exists():
            os.system(
                "git clone --depth 1 https://github.com/shai200/ShaiLearnsAI.git /content/ShaiLearnsAI"
            )
        return root
    raise RuntimeError(
        "Clone https://github.com/shai200/ShaiLearnsAI and run this notebook from that directory."
    )

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
print("repo", ROOT)

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from labs._shared.seed import seed_everything

seed_everything(0)


## Guided exercise: imbalanced classification

Ninety-five percent of the rows are the negative class. A classifier that always says negative looks accurate and never finds a positive.


In [ ]:
X, y = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.95, 0.05],
    class_sep=1.2,
    flip_y=0.01,
    random_state=0,
)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

majority = DummyClassifier(strategy="most_frequent").fit(Xtr_s, ytr)
logit = LogisticRegression(max_iter=500).fit(Xtr_s, ytr)

def classification_row(name, model):
    pred = model.predict(Xte_s)
    proba = model.predict_proba(Xte_s)[:, 1] if hasattr(model, "predict_proba") else pred
    return {
        "model": name,
        "accuracy": accuracy_score(yte, pred),
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
        "f1": f1_score(yte, pred, zero_division=0),
        "roc_auc": roc_auc_score(yte, proba),
    }

table = pd.DataFrame([
    classification_row("majority class", majority),
    classification_row("logistic regression", logit),
])
print(table.round(3).to_string(index=False))
assert table.loc[0, "accuracy"] > 0.9
assert table.loc[0, "f1"] < table.loc[1, "f1"]
assert table.loc[1, "roc_auc"] > 0.7

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv = cross_val_score(LogisticRegression(max_iter=500), scaler.transform(X), y, cv=folds, scoring="f1")
print("5-fold F1", np.round(cv, 3), "mean", round(float(cv.mean()), 3))
assert cv.mean() > table.loc[0, "f1"]


## Guided exercise: regression

Ridge regression on a synthetic linear target with noise. Mean squared error and R² are both reported. R² near 1 means the residual is small relative to the variance of `y`. It does not mean the model is useful outside this distribution.


In [ ]:
Xr, yr = make_regression(n_samples=400, n_features=6, noise=15.0, random_state=0)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.25, random_state=0)
reg = Ridge(alpha=1.0).fit(Xr_tr, yr_tr)
pred = reg.predict(Xr_te)
mse = mean_squared_error(yr_te, pred)
r2 = r2_score(yr_te, pred)
print(f"test MSE {mse:.2f}  R2 {r2:.3f}")
assert r2 > 0.8


## Your turn

Fit a depth-1 decision tree and an unrestricted tree on a small, wide training set. Compare training accuracy, test accuracy, and the gap between them.

## Solution


In [ ]:
Xo, yo = make_classification(
    n_samples=180, n_features=30, n_informative=6, n_redundant=0, random_state=1,
)
Xo_tr, Xo_te, yo_tr, yo_te = train_test_split(Xo, yo, test_size=0.4, random_state=1)
rows = []
for depth, model in (
    (1, DecisionTreeClassifier(max_depth=1, random_state=0)),
    (None, DecisionTreeClassifier(max_depth=None, random_state=0)),
):
    model.fit(Xo_tr, yo_tr)
    train_acc = model.score(Xo_tr, yo_tr)
    test_acc = model.score(Xo_te, yo_te)
    rows.append({"max_depth": str(depth), "train": train_acc, "test": test_acc, "gap": train_acc - test_acc})
overfit = pd.DataFrame(rows)
print(overfit.round(3).to_string(index=False))
assert overfit.loc[1, "train"] > overfit.loc[0, "train"]
assert overfit.loc[1, "gap"] > overfit.loc[0, "gap"]


## Beyond the course

Sweep the decision threshold on the logistic scores. A higher threshold predicts fewer positives. Precision and recall are not required to move in lockstep on a tiny test set, so the recorded result is the positive rate, plus the precision and recall at each threshold. The question underneath is which threshold the product can live with.


In [ ]:
scores = logit.predict_proba(Xte_s)[:, 1]
sweep = []
for threshold in (0.2, 0.5, 0.8):
    pred = (scores >= threshold).astype(int)
    sweep.append({
        "threshold": threshold,
        "predicted_positive_rate": float(pred.mean()),
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
    })
sweep_df = pd.DataFrame(sweep)
print(sweep_df.round(3).to_string(index=False))
assert sweep_df["predicted_positive_rate"].is_monotonic_decreasing

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(sweep_df["threshold"], sweep_df["precision"], marker="o", label="precision")
ax.plot(sweep_df["threshold"], sweep_df["recall"], marker="o", label="recall")
ax.set_xlabel("threshold")
ax.legend()
ax.set_title("Imbalanced test set")
plt.tight_layout()
plt.show()
print("LAB OK")


## Research questions

1. The majority classifier has high accuracy and zero F1. Name a product decision for which you would still ship the majority classifier, and one for which you would not.
2. Five-fold F1 moved between folds. How large a gap between folds would stop you from quoting the mean alone?
3. The unrestricted tree's training accuracy is not evidence about the test set. What would you change in the data, rather than in `max_depth`, to shrink that gap?

## Further reading

- scikit-learn user guide, sections "Supervised learning", "Model evaluation", and "Cross-validation" (BSD-3). https://scikit-learn.org/stable/supervised_learning.html
- Hastie, Tibshirani, and Friedman, *The Elements of Statistical Learning*, chapters 2 and 7. Reference, not copied.
